# Fact-Grounding Classifier: Live Training Reproduction

Trains the paper's `microsoft/deberta-v3-large` classifier on the official MuSiQue CSV split. Each notebook cell prints a timestamped checkpoint; long steps report progress while running. Training logs appear every 10 optimizer steps, validation prediction logs every 250 batches, and recoverable model checkpoints are saved each epoch with the best dev-F1 checkpoint retained. Kaggle Internet must be enabled for package, data, and base-model downloads.


In [ ]:
import sys, subprocess, json, platform, time, math, random, gc
from pathlib import Path
def checkpoint(name, status, **details):
    print(json.dumps({"time":time.strftime("%Y-%m-%d %H:%M:%S"),"checkpoint":name,"status":status,**details},default=str),flush=True)
checkpoint("00_environment_and_dependencies","begin")
pip_started=time.perf_counter()
subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers==4.57.1","sentencepiece","protobuf","scikit-learn","accelerate"])
import numpy as np, pandas as pd, torch, transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Select a Kaggle GPU accelerator."
gpus=[]
for i in range(torch.cuda.device_count()):
    free,total=torch.cuda.mem_get_info(i); p=torch.cuda.get_device_properties(i)
    gpus.append({"name":p.name,"total_gib":round(total/2**30,2),"free_at_start_gib":round(free/2**30,2)})
runtime={"python":sys.version,"platform":platform.platform(),"torch":torch.__version__,"transformers":transformers.__version__,"cuda":torch.version.cuda,"gpus":gpus,"seed":SEED}
checkpoint("00_environment_and_dependencies","complete",pip_seconds=round(time.perf_counter()-pip_started,2),gpu_count=len(gpus))
print(json.dumps(runtime,indent=2))

## Checkpoint 01 — Load and verify official splits

The files have `text,label` columns and no IDs. Kaggle input files are preferred. If the private input is not attached to this account, the notebook downloads the same files from the pinned official repository commit and verifies SHA256 before reading. Download progress is printed every 10 MiB. Exact duplicates are counted and retained. We check cross-split text leakage and conflicting labels. The formatter caps passage content at 1,500 characters; the full string also includes the question prefix.


In [ ]:
import hashlib, urllib.request
checkpoint("01_official_data","begin")
DATA_SPECS={
    "deberta_fg_train.csv":("https://raw.githubusercontent.com/RichardZhu123/fact-grounding-gap/3df77c2c94863ff35b398e14b37bd1e130e9d6de/deberta_fg_train.csv","7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D"),
    "deberta_fg_dev.csv":("https://raw.githubusercontent.com/RichardZhu123/fact-grounding-gap/3df77c2c94863ff35b398e14b37bd1e130e9d6de/deberta_fg_dev.csv","F7EC0B2308EBDC811BFC11828A076F498FD2CB542C503274DAD5CBA852EBD1BC"),
}
DATA_ROOT=Path('/kaggle/working/official_training_data'); DATA_ROOT.mkdir(parents=True,exist_ok=True)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
    return h.hexdigest().upper()
def find_file(name):
    hits=list(Path('/kaggle/input').rglob(name))
    url,expected=DATA_SPECS[name]
    if hits:
        path=hits[0]
    else:
        path=DATA_ROOT/name
        if not path.exists() or sha256_file(path)!=expected:
            checkpoint("01_official_data_download","begin",file=name,url=url)
            request=urllib.request.Request(url,headers={"User-Agent":"fact-grounding-reproduction/1.0"})
            with urllib.request.urlopen(request,timeout=60) as response, open(path,'wb') as dest:
                total=int(response.headers.get('Content-Length',0)); received=0; next_report=10*1024*1024
                while True:
                    chunk=response.read(1024*1024)
                    if not chunk: break
                    dest.write(chunk); received+=len(chunk)
                    if received>=next_report:
                        checkpoint("01_official_data_download","progress",file=name,downloaded_mib=round(received/2**20,1),total_mib=round(total/2**20,1) if total else None)
                        next_report+=10*1024*1024
            checkpoint("01_official_data_download","complete",file=name,bytes=path.stat().st_size)
    actual=sha256_file(path)
    if actual!=expected: raise ValueError(f"SHA256 mismatch for {path}: {actual} != {expected}")
    checkpoint("01_official_data_file_verified","complete",file=name,path=str(path),sha256=actual)
    return path
train_path=find_file('deberta_fg_train.csv'); dev_path=find_file('deberta_fg_dev.csv')
checkpoint("01_official_data","reading_csv")
train=pd.read_csv(train_path,dtype={'text':str,'label':int})
dev=pd.read_csv(dev_path,dtype={'text':str,'label':int})
assert list(train.columns)==['text','label'] and list(dev.columns)==['text','label']
assert len(train)==46610 and len(dev)==6404
def audit(frame):
    g=frame.groupby('text').label.agg(['count','nunique'])
    passage=frame.text.str.split('\\n\\nPassages:\\n',n=1).str[-1]
    return {'rows':len(frame),'unique_texts':int(frame.text.nunique()),'duplicate_rows':int((g['count']-1).clip(lower=0).sum()),'conflicting_duplicate_texts':int((g['nunique']>1).sum()),'labels':{str(k):int(v) for k,v in frame.label.value_counts().sort_index().items()},'max_full_text_chars':int(frame.text.str.len().max()),'max_passage_chars':int(passage.str.len().max())}
audit_data={'train':audit(train),'dev':audit(dev),'exact_train_dev_overlap':len(set(train.text)&set(dev.text)),'id_column_present':False}
print(json.dumps(audit_data,indent=2))
assert audit_data['exact_train_dev_overlap']==0
assert audit_data['train']['max_passage_chars']<=1500 and audit_data['dev']['max_passage_chars']<=1500
checkpoint("02_dataset_audit","complete",train_rows=len(train),dev_rows=len(dev),train_dev_overlap=audit_data['exact_train_dev_overlap'])


## Checkpoint 03 — Model, tokenizer, and 100-step GPU profile

The effective batch stays at 16. Kaggle reported two 14.56-GiB Tesla T4 GPUs. Earlier DataParallel profiles projected 11.39 hours through epoch 3 after dynamic padding, non-reentrant gradient checkpointing, and length-grouped batches. This run explicitly launches two `torchrun` processes so each T4 owns a distributed-training worker; it records world size, rank, effective batch, and per-GPU memory. Dynamic padding, length grouping, and the 512-token cap remain. PyTorch recommends the non-reentrant checkpoint implementation, which avoids limitations of the reentrant variant. Kaggle's observed notebook limit is 8 hours, so the notebook stops before full fine-tuning if its three-epoch projection plus 25% overhead exceeds 7 hours, leaving time for validation and checkpoint export. The paper reports convergence by epoch 3; if early stopping requires more epochs, the run may still exceed the platform limit.


In [ ]:
import sys, subprocess, json, math, re, os, queue, signal, threading, time
from pathlib import Path
WORKER_SOURCE = 'import argparse\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport transformers\nfrom transformers import (\n    AutoModelForSequenceClassification,\n    AutoTokenizer,\n    DataCollatorWithPadding,\n    EarlyStoppingCallback,\n    Trainer,\n    TrainerCallback,\n    TrainingArguments,\n)\nfrom sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support\n\n\nMODEL = "microsoft/deberta-v3-large"\nMAX_LENGTH = 512\nTARGET_BATCH = 16\nSEED = 42\nPAPER = {"precision": 0.805, "recall": 0.765, "f1": 0.785, "accuracy": 0.794}\n\n\ndef log_event(stage, rank=None, **fields):\n    rank_text = "?" if rank is None else str(rank)\n    payload = json.dumps(fields, ensure_ascii=False, default=str)\n    print(f"[{time.strftime(\'%Y-%m-%d %H:%M:%S\')}] [rank={rank_text}] {stage} {payload}", flush=True)\n\n\nclass SaveEventCallback(TrainerCallback):\n    def on_save(self, args, state, control, **kwargs):\n        if state.is_world_process_zero:\n            log_event("recovery_checkpoint_saved", rank=0, step=state.global_step, path=args.output_dir)\n        return control\n\n\nclass ProgressEventCallback(TrainerCallback):\n    def __init__(self):\n        self.prediction_steps = 0\n\n    def on_log(self, args, state, control, logs=None, **kwargs):\n        if state.is_world_process_zero:\n            log_event("training_progress", rank=0, step=state.global_step, total_steps=state.max_steps, logs=logs or {})\n        return control\n\n    def on_prediction_step(self, args, state, control, **kwargs):\n        self.prediction_steps += 1\n        if state.is_world_process_zero and self.prediction_steps % 250 == 0:\n            log_event("prediction_progress", rank=0, prediction_steps=self.prediction_steps)\n        return control\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--mode", choices=["profile", "full"], required=True)\n    parser.add_argument("--data-root", required=True)\n    parser.add_argument("--batch", type=int, required=True)\n    parser.add_argument("--accum", type=int, required=True)\n    parser.add_argument("--profile-out", default="/kaggle/working/profile_summary.json")\n    args = parser.parse_args()\n\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    world_size = int(os.environ.get("WORLD_SIZE", "1"))\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n    log_event("worker_started", rank, mode=args.mode, world_size=world_size, local_rank=local_rank, batch=args.batch, accumulation=args.accum)\n    train_path = Path(args.data_root) / "deberta_fg_train.csv"\n    dev_path = Path(args.data_root) / "deberta_fg_dev.csv"\n    data_start = time.perf_counter()\n    log_event("csv_load_started", rank, train_path=train_path, dev_path=dev_path)\n    train = pd.read_csv(train_path, dtype={"text": str, "label": int})\n    dev = pd.read_csv(dev_path, dtype={"text": str, "label": int})\n    if len(train) != 46610 or len(dev) != 6404:\n        raise ValueError(f"Unexpected official split sizes: {len(train)} train, {len(dev)} dev")\n    log_event("csv_load_completed", rank, train_rows=len(train), dev_rows=len(dev), seconds=round(time.perf_counter() - data_start, 2))\n\n    tokenizer_start = time.perf_counter()\n    log_event("tokenizer_load_started", rank, model=MODEL)\n    tokenizer = AutoTokenizer.from_pretrained(MODEL)\n    log_event("tokenizer_load_completed", rank, seconds=round(time.perf_counter() - tokenizer_start, 2))\n\n    class Encoded(torch.utils.data.Dataset):\n        def __init__(self, frame, name):\n            start = time.perf_counter()\n            log_event("tokenization_started", rank, dataset=name, examples=len(frame), max_length=MAX_LENGTH)\n            self.x = None\n            texts = frame.text.tolist()\n            for offset in range(0, len(texts), 512):\n                encoded = tokenizer(texts[offset:offset + 512], max_length=MAX_LENGTH, padding=False, truncation=True)\n                if self.x is None:\n                    self.x = {key: [] for key in encoded}\n                for key in self.x:\n                    self.x[key].extend(encoded[key])\n                if (offset // 512) % 10 == 0 or offset + 512 >= len(texts):\n                    log_event("tokenization_progress", rank, dataset=name, processed=min(offset + 512, len(texts)), total=len(texts), elapsed_seconds=round(time.perf_counter() - start, 2))\n            self.lengths = np.asarray([len(ids) for ids in self.x["input_ids"]], dtype=np.int32)\n            self.y = torch.tensor(frame.label.to_numpy(), dtype=torch.long)\n            log_event("tokenization_completed", rank, dataset=name, examples=len(self.y), seconds=round(time.perf_counter() - start, 2))\n\n        def __len__(self):\n            return len(self.y)\n\n        def __getitem__(self, i):\n            return {\n                **{k: torch.tensor(v[i], dtype=torch.long) for k, v in self.x.items()},\n                "labels": self.y[i],\n            }\n\n    train_ds = Encoded(train, "train")\n    dev_ds = Encoded(dev, "dev")\n    collator = DataCollatorWithPadding(tokenizer, padding="longest", pad_to_multiple_of=8)\n    token_length_summary = {\n        name: {\n            "mean": float(np.mean(ds.lengths)),\n            "p50": float(np.percentile(ds.lengths, 50)),\n            "p90": float(np.percentile(ds.lengths, 90)),\n            "p99": float(np.percentile(ds.lengths, 99)),\n            "max": int(np.max(ds.lengths)),\n        }\n        for name, ds in [("train", train_ds), ("dev", dev_ds)]\n    }\n\n    actual_batch = args.batch * world_size * args.accum\n    if actual_batch != TARGET_BATCH:\n        raise ValueError(f"Worker effective batch {actual_batch} differs from target {TARGET_BATCH}")\n\n    def model_init():\n        start = time.perf_counter()\n        log_event("model_load_started", rank, model=MODEL)\n        model = AutoModelForSequenceClassification.from_pretrained(\n            MODEL,\n            num_labels=2,\n            id2label={0: "NOT-ANSWERABLE", 1: "ANSWERABLE"},\n            label2id={"NOT-ANSWERABLE": 0, "ANSWERABLE": 1},\n        )\n        log_event("model_load_completed", rank, parameters=sum(p.numel() for p in model.parameters()), seconds=round(time.perf_counter() - start, 2))\n        return model\n\n    def metric_fn(p):\n        y = p.label_ids\n        probability = torch.softmax(torch.tensor(p.predictions), dim=-1).numpy()[:, 1]\n        yhat = (probability >= 0.5).astype(int)\n        pr, re, f1, _ = precision_recall_fscore_support(\n            y, yhat, average="binary", pos_label=1, zero_division=0\n        )\n        return {"accuracy": accuracy_score(y, yhat), "precision": pr, "recall": re, "f1": f1}\n\n    profile = args.mode == "profile"\n    output_dir = f"/kaggle/working/profile_ddp_{args.batch}" if profile else "/kaggle/working/training_runs_ddp"\n    import inspect\n\n    strategy_arg = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments).parameters else "evaluation_strategy"\n    kwargs = {\n        "output_dir": output_dir,\n        "num_train_epochs": 10,\n        "per_device_train_batch_size": args.batch,\n        "per_device_eval_batch_size": args.batch,\n        "gradient_accumulation_steps": args.accum,\n        "learning_rate": 2e-5,\n        "weight_decay": 0.01,\n        "warmup_ratio": 0.1,\n        "lr_scheduler_type": "linear",\n        "save_strategy": "epoch" if not profile else "no",\n        "load_best_model_at_end": not profile,\n        "metric_for_best_model": "f1" if not profile else None,\n        "greater_is_better": True,\n        "save_total_limit": 2,\n        "fp16": True,\n        "gradient_checkpointing": True,\n        "gradient_checkpointing_kwargs": {"use_reentrant": False},\n        "group_by_length": True,\n        "optim": "adamw_torch",\n        "logging_steps": 10,\n        "logging_first_step": True,\n        "disable_tqdm": True,\n        "log_level": "info",\n        "report_to": "none",\n        "seed": SEED,\n        "data_seed": SEED,\n        "dataloader_num_workers": 2,\n        "max_steps": 100 if profile else -1,\n    }\n    kwargs[strategy_arg] = "no" if profile else "epoch"\n    training_args = TrainingArguments(**kwargs)\n    log_event("trainer_ready", rank, mode=args.mode, world_size=world_size, actual_effective_batch=actual_batch, output_dir=output_dir, max_steps=training_args.max_steps, logging_steps=training_args.logging_steps, save_strategy=str(training_args.save_strategy), eval_strategy=str(getattr(training_args, strategy_arg)))\n    trainer = Trainer(\n        model=model_init(),\n        args=training_args,\n        train_dataset=train_ds,\n        eval_dataset=None if profile else dev_ds,\n        compute_metrics=None if profile else metric_fn,\n        callbacks=[] if profile else [EarlyStoppingCallback(early_stopping_patience=3), SaveEventCallback(), ProgressEventCallback()],\n        data_collator=collator,\n    )\n\n    parallelism = {\n        "rank": rank,\n        "local_rank": local_rank,\n        "world_size": world_size,\n        "n_gpu_seen_by_trainer": int(training_args.n_gpu),\n        "per_device_batch_size": args.batch,\n        "train_batch_size": int(training_args.train_batch_size),\n        "gradient_accumulation": args.accum,\n        "actual_effective_batch": actual_batch,\n        "parallel_mode": str(training_args.parallel_mode),\n        "device": str(training_args.device),\n    }\n\n    torch.cuda.reset_peak_memory_stats(local_rank)\n    start = time.perf_counter()\n    log_event("train_started", rank, mode=args.mode, max_steps=training_args.max_steps, dataset_examples=len(train_ds))\n    train_output = trainer.train()\n    torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    log_event("train_completed", rank, mode=args.mode, seconds=round(elapsed, 2), global_step=trainer.state.global_step, metrics=train_output.metrics)\n    local_peak = torch.tensor([torch.cuda.max_memory_allocated(local_rank) / 2**30], device=training_args.device)\n    if dist.is_initialized():\n        gathered = [torch.zeros_like(local_peak) for _ in range(world_size)]\n        dist.all_gather(gathered, local_peak)\n        peaks = [round(float(item.item()), 3) for item in gathered]\n        max_elapsed = torch.tensor([elapsed], device=training_args.device)\n        dist.all_reduce(max_elapsed, op=dist.ReduceOp.MAX)\n        elapsed = float(max_elapsed.item())\n    else:\n        peaks = [round(float(local_peak.item()), 3)]\n\n    if profile:\n        if rank == 0:\n            result = {\n                "optimizer_steps": 100,\n                "elapsed_seconds": elapsed,\n                "seconds_per_step": elapsed / 100,\n                "micro_batch_per_gpu": args.batch,\n                "gradient_accumulation": args.accum,\n                "gpu_count": world_size,\n                "effective_batch": actual_batch,\n                "parallelism": parallelism,\n                "peak_allocated_gib_per_gpu": peaks,\n                "gradient_checkpointing": True,\n                "gradient_checkpointing_kwargs": {"use_reentrant": False},\n                "dynamic_padding": True,\n                "group_by_length": True,\n                "token_length_summary": token_length_summary,\n                "torch_version": torch.__version__,\n                "transformers_version": transformers.__version__,\n            }\n            Path(args.profile_out).write_text(json.dumps(result, indent=2), encoding="utf-8")\n            print(json.dumps(result, indent=2))\n        if dist.is_initialized():\n            dist.barrier()\n        return\n\n    train_metrics = train_output.metrics\n    log_event("dev_evaluation_started", rank, examples=len(dev_ds))\n    eval_metrics = trainer.evaluate(dev_ds)\n    log_event("dev_evaluation_completed", rank, metrics=eval_metrics)\n    log_event("dev_prediction_started", rank, examples=len(dev_ds))\n    predictions = trainer.predict(dev_ds)\n    log_event("dev_prediction_completed", rank, metrics=predictions.metrics)\n    probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()[:, 1]\n    yhat = (probs >= 0.5).astype(int)\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        dev.label, yhat, average="binary", pos_label=1, zero_division=0\n    )\n    metrics = {\n        "dataset": "MuSiQue dev",\n        "n": len(dev),\n        "n_positive": int(dev.label.sum()),\n        "n_negative": int((dev.label == 0).sum()),\n        "threshold": 0.5,\n        "accuracy": float(accuracy_score(dev.label, yhat)),\n        "precision": float(precision),\n        "recall": float(recall),\n        "f1": float(f1),\n        "confusion_matrix_labels_0_1": confusion_matrix(dev.label, yhat, labels=[0, 1]).tolist(),\n        "training_seconds": float(train_metrics.get("train_runtime", elapsed)),\n        "best_checkpoint": trainer.state.best_model_checkpoint,\n        "best_f1": trainer.state.best_metric,\n        "global_step": trainer.state.global_step,\n    }\n    paper_comparison = {k: {"paper": v, "reproduced": metrics[k], "delta": metrics[k] - v} for k, v in PAPER.items()}\n    output = Path("/kaggle/working/deberta_fact_grounding_checkpoint")\n    checkpoint_export_seconds = None\n    log_event("best_checkpoint_export_started", rank, output=output)\n    if rank == 0:\n        output.mkdir(parents=True, exist_ok=True)\n    if dist.is_initialized():\n        dist.barrier()\n    export_start = time.perf_counter()\n    trainer.save_model(str(output))\n    if rank == 0:\n        tokenizer.save_pretrained(str(output))\n        checkpoint_export_seconds = time.perf_counter() - export_start\n        log_event("best_checkpoint_export_completed", rank, seconds=round(checkpoint_export_seconds, 2))\n    if dist.is_initialized():\n        dist.barrier()\n    if rank == 0:\n        pred = dev.copy()\n        pred["prob_answerable"] = probs\n        pred["prediction"] = yhat\n        pred.to_csv("/kaggle/working/in_domain_predictions.csv", index=False)\n        pd.DataFrame([metrics]).to_csv("/kaggle/working/in_domain_metrics.csv", index=False)\n        Path("/kaggle/working/in_domain_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")\n        Path("/kaggle/working/training_log.json").write_text(json.dumps(trainer.state.log_history, indent=2), encoding="utf-8")\n\n    if dist.is_initialized():\n        dist.barrier()\n\n    reload_check = None\n    reload_seconds = None\n    if rank == 0:\n        log_event("fresh_reload_check_started", rank, output=output)\n        reload_start = time.perf_counter()\n        fresh_tokenizer = AutoTokenizer.from_pretrained(output)\n        fresh_model = AutoModelForSequenceClassification.from_pretrained(output).cuda().eval()\n        # Check a spread of examples in small chunks so that the original and\n        # freshly loaded models do not create a large validation activation.\n        check_indices = np.linspace(0, len(dev) - 1, num=min(32, len(dev)), dtype=int).tolist()\n        trainer.model.eval()\n        original_chunks, restored_chunks = [], []\n        with torch.no_grad():\n            for idx in check_indices:\n                sample = fresh_tokenizer(\n                    dev.text.iloc[idx], max_length=MAX_LENGTH, padding="max_length", truncation=True, return_tensors="pt"\n                ).to("cuda")\n                original_chunks.append(trainer.model(**sample).logits.float().cpu())\n                restored_chunks.append(fresh_model(**sample).logits.float().cpu())\n        original = torch.cat(original_chunks, dim=0)\n        restored = torch.cat(restored_chunks, dim=0)\n        max_abs_logit_diff = float((original - restored).abs().max())\n        same_predictions = torch.equal(original.argmax(-1), restored.argmax(-1))\n        # CUDA kernels and the saved/reloaded fp32 weights can differ slightly\n        # numerically; prediction identity and a 1e-3 logit tolerance are the\n        # useful deployment check. The previous 1e-5 tolerance rejected an\n        # otherwise identical checkpoint (observed max difference: 2.9e-4).\n        logit_tolerance = 1e-3\n        reload_check = {\n            "checkpoint_dir": str(output),\n            "files": sorted(p.name for p in output.iterdir()),\n            "sample_count": len(check_indices),\n            "sample_indices": check_indices,\n            "max_abs_logit_diff": max_abs_logit_diff,\n            "logit_tolerance": logit_tolerance,\n            "same_prediction_all_samples": bool(same_predictions),\n            "pass": bool(same_predictions and max_abs_logit_diff <= logit_tolerance),\n        }\n        reload_seconds = time.perf_counter() - reload_start\n        log_event("fresh_reload_check_completed", rank, result=reload_check, seconds=round(reload_seconds, 2))\n        metadata = {\n            "base_model": MODEL,\n            "parameter_count": sum(p.numel() for p in trainer.model.parameters()),\n            "max_length": MAX_LENGTH,\n            "label_mapping": {"0": "NOT-ANSWERABLE", "1": "ANSWERABLE"},\n            "threshold": 0.5,\n            "seed": SEED,\n            "train_examples": len(train),\n            "dev_examples": len(dev),\n            "training_configuration": {\n                "learning_rate": 2e-5,\n                "effective_batch": TARGET_BATCH,\n                "micro_batch_per_gpu": args.batch,\n                "gradient_accumulation": args.accum,\n                "max_epochs": 10,\n                "early_stopping_patience": 3,\n                "warmup_ratio": 0.1,\n                "scheduler": "linear",\n                "weight_decay": 0.01,\n                "precision": "fp16",\n                "gradient_checkpointing": True,\n                "gradient_checkpointing_kwargs": {"use_reentrant": False},\n                "dynamic_padding": True,\n                "group_by_length": True,\n            },\n            "training_arguments": json.loads(json.dumps(training_args.to_dict(), default=str)),\n            "parallelism": parallelism,\n            "token_length_summary": token_length_summary,\n            "training_runtime_metrics": train_metrics,\n            "evaluation_metrics": eval_metrics,\n            "validation_metrics": metrics,\n            "paper_comparison": paper_comparison,\n            "checkpoint_reload": reload_check,\n            "checkpoint_export_seconds": checkpoint_export_seconds,\n            "checkpoint_reload_seconds": reload_seconds,\n            "peak_allocated_gib_per_gpu": peaks,\n            "versions": {"torch": torch.__version__, "transformers": transformers.__version__},\n        }\n        (output / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n        (output / "training_args.json").write_text(json.dumps(metadata["training_arguments"], indent=2), encoding="utf-8")\n        (output / "label_mapping.json").write_text(json.dumps(metadata["label_mapping"], indent=2), encoding="utf-8")\n        Path("/kaggle/working/reproduction_summary.json").write_text(\n            json.dumps(\n                {\n                    "metrics": metrics,\n                    "comparison": paper_comparison,\n                    "reload_check": reload_check,\n                    "training_runtime_metrics": train_metrics,\n                    "checkpoint_export_seconds": checkpoint_export_seconds,\n                    "checkpoint_reload_seconds": reload_seconds,\n                    "gpu_configuration": parallelism,\n                },\n                indent=2,\n            ),\n            encoding="utf-8",\n        )\n        # Persist diagnostics before raising so a failed validation never\n        # hides the trained checkpoint\'s metrics or reload evidence.\n        (output / "checkpoint_reload.json").write_text(json.dumps(reload_check, indent=2), encoding="utf-8")\n        print(json.dumps(metadata, indent=2))\n        if not reload_check["pass"]:\n            raise RuntimeError(f"Fresh checkpoint reload failed; diagnostics were saved: {reload_check}")\n\n    if dist.is_initialized():\n        dist.barrier()\n\n\nif __name__ == "__main__":\n    main()\n'
worker_path=Path('/kaggle/working/kaggle_train_worker.py')
worker_path.write_text(WORKER_SOURCE,encoding='utf-8')

def run_streaming(cmd,label,log_path,heartbeat_seconds=30,timeout_seconds=None):
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env['TOKENIZERS_PARALLELISM']='false'
    started=time.perf_counter(); lines=[]
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] [{label}] PROCESS_START cmd={cmd}',flush=True)
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,encoding='utf-8',errors='replace',env=env,start_new_session=True)
    messages=queue.Queue(); finished=object()
    def pump_output():
        try:
            for line in process.stdout: messages.put(line)
        finally:
            messages.put(finished)
    reader=threading.Thread(target=pump_output,daemon=True); reader.start()
    log_path=Path(log_path); done=False
    with log_path.open('w',encoding='utf-8') as sink:
        while not done:
            elapsed=time.perf_counter()-started
            if timeout_seconds is not None and elapsed>timeout_seconds:
                print(f'[{label}] TIMEOUT after {elapsed/60:.1f} minutes; terminating process group',flush=True)
                try: os.killpg(process.pid,signal.SIGTERM)
                except OSError: process.terminate()
                try: process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    try: os.killpg(process.pid,signal.SIGKILL)
                    except OSError: process.kill()
                    process.wait()
                raise TimeoutError(f'{label} exceeded {timeout_seconds} seconds; inspect {log_path}')
            try:
                item=messages.get(timeout=heartbeat_seconds)
            except queue.Empty:
                print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] [{label}] HEARTBEAT elapsed={elapsed/60:.1f}m child_alive={process.poll() is None}',flush=True)
                continue
            if item is finished:
                done=True
            else:
                lines.append(item); sink.write(item); sink.flush(); print(item,end='',flush=True)
    returncode=process.wait(); elapsed=time.perf_counter()-started
    print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] [{label}] PROCESS_END returncode={returncode} elapsed={elapsed/60:.1f}m log={log_path}',flush=True)
    return returncode,''.join(lines)

train_file=Path(find_file('deberta_fg_train.csv')); dev_file=Path(find_file('deberta_fg_dev.csv'))
assert train_file.parent==dev_file.parent
DATA_ROOT=train_file.parent
TARGET_BATCH=16; ngpu=torch.cuda.device_count()
if ngpu<1: raise RuntimeError('No Kaggle GPU detected for DDP training.')
profiles=[]; prof=None
profile_batches=list(dict.fromkeys([min(8,max(1,TARGET_BATCH//ngpu)),min(4,max(1,TARGET_BATCH//ngpu)),min(2,max(1,TARGET_BATCH//ngpu)),1]))
for batch in profile_batches:
    accumulation=max(1,math.ceil(TARGET_BATCH/(batch*ngpu)))
    profile_path=Path(f'/kaggle/working/profile_batch_{batch}.json')
    cmd=[sys.executable,'-u','-m','torch.distributed.run','--standalone',f'--nproc_per_node={ngpu}',str(worker_path),'--mode','profile','--data-root',str(DATA_ROOT),'--batch',str(batch),'--accum',str(accumulation),'--profile-out',str(profile_path)]
    print(f'PROFILE_CHECKPOINT_BEGIN batch={batch} accumulation={accumulation} effective_batch={batch*ngpu*accumulation}',flush=True)
    returncode,combined=run_streaming(cmd,f'profile-batch-{batch}',f'/kaggle/working/profile_batch_{batch}.log',heartbeat_seconds=30,timeout_seconds=1800)
    print(f'PROFILE_CHECKPOINT_END batch={batch} accumulation={accumulation} returncode={returncode}',flush=True)
    if returncode==0 and profile_path.exists():
        prof=json.loads(profile_path.read_text(encoding='utf-8')); print(json.dumps(prof,indent=2)); micro,accum=batch,accumulation; break
    if re.search(r'out of memory|cuda out of memory|outofmemoryerror',combined,re.I):
        print(f'OOM at per-device batch {batch}; retrying smaller batch.')
        continue
    raise RuntimeError('DDP profile failed for a non-OOM reason. See the captured profile log: '+combined[-5000:])
if prof is None: raise RuntimeError(f'DDP GPU profiling failed at all per-device batches {profile_batches}; no full training was started.')
steps_epoch=math.ceil(len(train)/TARGET_BATCH); estimate=prof['seconds_per_step']*steps_epoch*3*1.25/3600
prof.update({'steps_per_epoch':steps_epoch,'estimated_steps_through_epoch_3':steps_epoch*3,'estimated_train_hours_with_25pct_overhead':estimate})
Path('/kaggle/working/profile_summary.json').write_text(json.dumps(prof,indent=2),encoding='utf-8')
print(json.dumps(prof,indent=2))
if estimate>7: raise RuntimeError('DDP profile projects >7 hours through epoch 3 including 25% overhead; stop before full training under the observed Kaggle 8-hour limit.')


## Checkpoint 04 — Full training and complete dev evaluation

The official code uses an epoch cap of 10, 2e-5 learning rate, effective batch 16, 512 tokens, 10% linear warmup, and dev F1 with patience 3. The paper reports convergence at epoch 3. No examples are removed and no epoch cap is reduced. The external worker uses `torchrun` DistributedDataParallel to use both Kaggle T4 GPUs explicitly.


In [ ]:
import sys, subprocess, json
from pathlib import Path
profile=json.loads(Path('/kaggle/working/profile_summary.json').read_text(encoding='utf-8'))
worker_path=Path('/kaggle/working/kaggle_train_worker.py')
train_file=Path(find_file('deberta_fg_train.csv')); dev_file=Path(find_file('deberta_fg_dev.csv'))
assert train_file.parent==dev_file.parent
micro=int(profile['micro_batch_per_gpu']); accum=int(profile['gradient_accumulation']); ngpu=int(profile['gpu_count'])
cmd=[sys.executable,'-u','-m','torch.distributed.run','--standalone',f'--nproc_per_node={ngpu}',str(worker_path),'--mode','full','--data-root',str(train_file.parent),'--batch',str(micro),'--accum',str(accum)]
training_log=Path('/kaggle/working/ddp_training_output.log')
print('CHECKPOINT full_training_begin',json.dumps({'time':time.strftime('%Y-%m-%d %H:%M:%S'),'micro_batch_per_gpu':micro,'gradient_accumulation':accum,'gpu_count':ngpu,'effective_batch':micro*accum*ngpu}),flush=True)
returncode,full_output=run_streaming(cmd,'full-training',training_log,heartbeat_seconds=30,timeout_seconds=None)
if returncode: raise RuntimeError('DDP full training failed; inspect /kaggle/working/ddp_training_output.log\n'+full_output[-5000:])
print('CHECKPOINT full_training_complete',flush=True)
out=Path('/kaggle/working/deberta_fact_grounding_checkpoint')
print('CHECKPOINT export_and_fresh_reload_begin',flush=True)
meta=json.loads((out/'training_metadata.json').read_text(encoding='utf-8'))
meta['data_audit']=audit_data; meta['runtime']=runtime; meta['profile']=profile
(out/'training_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
print(json.dumps(meta,indent=2))
print('CHECKPOINT export_and_fresh_reload_complete',flush=True)


## Checkpoint 05 — Export, fresh reload, and paper comparison

The threshold stays at 0.5. A fresh model reload is checked on 32 evenly spaced dev examples. Predictions must match exactly and maximum absolute logit difference must be at most 1e-3. The report and all metrics are saved before the check can fail, so diagnostics remain available. All key outputs are written under `/kaggle/working` so Kaggle keeps them with the notebook version.


In [ ]:
import json
from pathlib import Path
OUT=Path('/kaggle/working/deberta_fact_grounding_checkpoint')
metrics=json.loads(Path('/kaggle/working/in_domain_metrics.json').read_text(encoding='utf-8'))
summary=json.loads(Path('/kaggle/working/reproduction_summary.json').read_text(encoding='utf-8'))
reload_check=summary['reload_check']; comparison=summary['comparison']
assert reload_check['pass'],reload_check
assert len(pd.read_csv('/kaggle/working/in_domain_predictions.csv'))==6404
assert (OUT/'training_metadata.json').exists() and (OUT/'checkpoint_reload.json').exists() and (OUT/'config.json').exists()
print(json.dumps({'metrics':metrics,'paper_comparison':comparison,'checkpoint_reload':reload_check,'checkpoint_files':sorted(p.name for p in OUT.iterdir()),'checkpoint_export_seconds':summary.get('checkpoint_export_seconds'),'checkpoint_reload_seconds':summary.get('checkpoint_reload_seconds')},indent=2))
